### Project Name : **AI Pilot** model

- First let's read our [*CSV*](SmartRocketTrainingData20260507_191122.csv) data file 
We don't use Pandas or Pytorch because we want to understand the under the hood concepts of trining the models this the bigger idea we work for here

In [389]:
import csv

file_path = "SmartRocketTrainingData20260507_191122.csv"
data = []

with open(file_path,'r') as file:
    reader = csv.reader(file)
    header = next(reader)

    for row in reader:
        try:
            presed_row_value = [float(row_value) for row_value in row]
            data.append(presed_row_value)
        except ValueError:
            continue


Now let's filter the Data in inputs X and outputs Y

In [390]:
X,Y = [],[]
for row in data:
    if row[2] == -1: continue

    inputs = [
        row[3], # DistanceX
        row[4], # DistanceY
        row[7], # VelocityX
        row[8], # VelocityY
        row[9], # Angle
        row[10] # AngularVelocity
    ]
    inputs[0] = inputs[0] / 60.0  # div by the max for all
    inputs[1] = inputs[1] / 60.0
    inputs[2] = inputs[2] / 15.0
    inputs[3] = inputs[3] / 15.0
    inputs[4] = inputs[4] / 180
    inputs[5] = inputs[5] / 30

    labels = [
        row[0], # Speed -> Engine thrust
        row[1]  # Steering -> rocket direction
    ]

    labels[0] = labels[0] / 30.0

    X.append(inputs)
    Y.append(labels)


In [391]:
from engine import Value
import random

In [392]:

class Neuron:
    def __init__(self,nin):  # nin -> n input
        self.weight = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.bias = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi * xi for wi , xi in zip(self.weight,x)),self.bias) # (wight * data) + bias
        return act.tanh() # Activation Functions
    
    def parameters(self):
        return self.weight + [self.bias]
    
class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    
    def parameters(self):
        return [ p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self,nin,nout):
        sz = [nin] + nout
        self.layers = [Layer(sz[i],sz[i+1]) for i in range(len(nout))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [parameter for p in self.layers for parameter in p.parameters()]
        

### Sanity Check

In [311]:
xs = [
  [0.2, 0.5, -0.1, 0.8, 0.0, -0.5], 
  [0.9, -0.9, 0.5, 0.1, 0.2, 0.8]   
]

ys = [
  [1.0, 0.0],  
  [-1.0, 1.0]  
]
model = MLP(6,[16,16,2])
for x in xs:
  model(x)
  
len(model.parameters())

418

In [ ]:
LR = -0.01 # Learning Rate
for iLoop in range(10000):
    ypred = [model(x) for x in xs] 
    loss = sum((yout - yget)**2 for y1 , y2 in zip(ys,ypred) for yget , yout in zip(y1,y2))
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()
    for p in model.parameters():
        p.data += LR * p.grad
    print(iLoop,"  ",loss.data)

0    0.26337698776060003
1    0.24631440356935275
2    0.2296307883706697
3    0.21340214847545352
4    0.1976989658832927
5    0.18258456280463703
6    0.1681137212207292
7    0.1543316284884143
8    0.1412731962637672
9    0.12896277252440547
10    0.11741423847168544
11    0.10663145674668648
12    0.096609017243908
13    0.0873332134425196
14    0.07878317610083037
15    0.07093209188322627
16    0.0637484407709436
17    0.05719719628976042
18    0.05124094491453741
19    0.04584089389285059
20    0.04095774890906388
21    0.03655245362788593
22    0.03258679174559584
23    0.02902385860041083
24    0.02582841375710164
25    0.022967128545522502
26    0.020408743635300673
27    0.01812415172323566
28    0.01608641962616769
29    0.01427076279388816
30    0.012654483711794106
31    0.011216884023188354
32    0.009939158587605914
33    0.00880427818328958
34    0.007796866204771146
35    0.006903073521509884
36    0.006110454654543601
37    0.0054078475875302635
38    0.0047852588421

In [107]:
[[yy.data for yy in y] for y in ypred]


[[0.9986639507439349, -1.3248433125005435e-06],
 [-0.9999930011368631, 0.998612912289003]]

### Save Result as [CSV](Ai-Pilot_SaveData.csv) file

In [328]:
save_path = "Ai-Pilot_SaveData.csv"

with open(save_path,"w",newline='') as save_csv:
    writer = csv.writer(save_csv,delimiter=',')
    data = [[yy.data for yy in y] for y in ypred]
    for dataLine in data:
        print(dataLine)
        writer.writerow(dataLine)

[-0.0007532713450463511, 0.001072737118095538]
[-0.0008018101785604881, 0.0010532406912863485]
[-0.0008592046996735258, 0.0010729882572905875]
[-0.0008841051132604325, 0.0010832440498500019]
[-0.0008936716331255796, 0.0010876571365823263]
[-0.000901694214017563, 0.0010915697983861993]
[-0.0009084034132295579, 0.0010949571327012696]
[-0.000914003677522766, 0.0010978844591028132]
[-0.0009177488153013838, 0.0010995634197828548]
[-0.0009210713061745906, 0.0011012254660803255]
[-0.0009239850848432292, 0.0011027845118480731]
[-0.0009265288117526421, 0.0011042263485127972]
[-0.0009287323571370424, 0.0011055339868410725]
[-0.0009306283935970322, 0.0011067010815987621]
[-0.0009322581080657488, 0.0011077378853120209]
[-0.0009336648482862182, 0.0011086632661842304]
[-0.000934859866435222, 0.001109463426687373]
[-0.0009358762669845613, 0.0011101581812499057]
[-0.0009367455695764372, 0.001110765061628534]
[-0.0009374801881206087, 0.001111285494988908]
[-0.0009381001311208155, 0.0011117299715979639]

### Now The really trining

In [394]:
add_size = 60

model = MLP(6,[16,16,2])
len(model.parameters())

418

In [395]:
for i in range(0,len(X),60):
    x = []
    y = []
    for i in range(i,i+add_size):
        x.append(X[i])
        y.append(Y[i])
 


    LR = -0.0001 # Learning Rate
    for iLoop in range(60): 
        ypred = [model(xi) for xi in x] 
        loss = sum((yout - yget)**2 for y1 , y2 in zip(y,ypred) for yget , yout in zip(y1,y2))
        for p in model.parameters():
            p.grad = 0.0
        loss.backward()
        for p in model.parameters():
            p.data += LR * p.grad
        print(iLoop,"  ",loss.data)

0    97.60468116969525
1    96.18454101736086
2    94.53445531785901
3    92.59570900928423
4    90.28993292495204
5    87.512078991245
6    84.12224530768042
7    79.9394637273466
8    74.74784220187207
9    68.34229212645589
10    60.66604210999469
11    52.07456168764248
12    43.54145208370892
13    36.264568497600315
14    30.700718255000297
15    26.364711148862877
16    22.596962026226976
17    19.04369987958635
18    15.632875959732772
19    12.4387367241022
20    9.579883404764624
21    7.153105167064839
22    5.198659648635196
23    3.696713243759793
24    2.5858521139018036
25    1.7877996993286256
26    1.2263437902400898
27    0.8370240359920859
28    0.5696998663057251
29    0.38734277092182356
30    0.26348765909350763
31    0.17960939477388466
32    0.1229132706476656
33    0.08463890519594147
34    0.05882217605289195
35    0.04141781524257072
36    0.029688796784213073
37    0.02178625082414576
38    0.016462589263376987
39    0.012876544983315549
40    0.010461094867

KeyboardInterrupt: 

In [324]:
model.layers

In [358]:
import json

In [404]:
save_wight_path = "Ai-Piloat_Weights.json"
def export_brain(model):
    print("--- Copy weights to Json file ---")
    model_brain = []
    for i ,layer in enumerate(model.layers):
        for j , neuron in enumerate(layer.neurons):
            layer_data = {
                "weights" : [w.data for w in layer.neurons[j].weight],
                "bias" : neuron.bias.data
            }
            print([w.data for w in layer.neurons[j].weight])
            model_brain.append(layer_data)
    with open(save_wight_path,"w") as save_json:
        json.dump(model_brain,save_json)
    print(len(model_brain))
export_brain(model)

--- Copy weights to Json file ---
[0.07535985450426193, -0.49974271057015907, 0.19048762336059716, 0.5436205839111302, -0.14800972374594973, -0.8077646964994691]
[-0.508903037298761, 0.18650196377952635, -0.6948947535074838, -0.4416792738827003, 0.102114132739158, -0.23264652601602576]
[-0.8930622037218395, 0.4044740766639448, 0.43018185677985543, 0.8312450327186477, -0.8220908611221746, -0.8471233218328145]
[0.6381313332425114, 0.9736145772506607, -0.3318687595716174, 0.7570920623870383, 0.6665546110415012, -0.2970359813712786]
[-0.22311231767729559, -0.17086710230070284, -0.11013493834745447, -0.8600911402433524, 0.5878155069164793, 0.5293394288704585]
[-0.34265921547162925, -0.14367265455730577, -0.5501619709737761, -0.5735173737029865, 0.6807802584040163, 0.18206702869237223]
[-0.36696185567086176, 0.5438446412704692, 0.2801407795869661, -0.34468424538567055, -0.07069648946707689, 0.23189253410897304]
[0.5508031878786888, -0.29081895109234085, 0.43152337603583757, -0.79192833694945